# LLM 서비스 개발 과정 — Day 4 (2026-03-27, 금)

## Memory — 대화 맥락(Chat History) 관리

> **모두의연구소 재직자 LLM 6기 · 3주차 Day 4**
> *LLM은 stateless! 대화 기억은 개발자가 직접 관리해야 합니다.*

---

### 오늘 배울 것 한눈에

| 파트 | 주제 | 핵심 |
|------|------|------|
| **1** | LLM이 왜 기억을 못하나 | stateless — 매 요청이 독립 |
| **2** | 가장 원시적인 방법 | `messages.append()` 로 무한 누적 |
| **3** | 토큰 비용 측정 | `tiktoken`으로 턴마다 토큰 계산 |
| **4** | `ConversationBufferMemory` (구형) | deprecated지만 개념 이해용 |
| **5** | `InMemoryChatMessageHistory` (모던) | LangChain 현재 표준 |
| **6** | `RunnableWithMessageHistory` | LCEL 체인에 메모리 자동 연결 |
| **7** | Window 메모리 | 최근 k턴만 유지 (`trim_messages`) |
| **8** | Summary 메모리 | 오래된 대화는 LLM이 요약 |
| **9** | 실전: `SummaryChatbot` 클래스 | Summary 메모리 + 프롬프트 + 체인 |

### 오늘의 한 줄 비유
> LLM은 **"매번 기억상실증인 상담원"** 입니다. 매 요청마다 "저 처음 뵙겠습니다"예요.
> 그래서 **내가(=개발자가) 매번 이전 대화를 요약해서 건네야** 상담이 이어집니다.
> Memory 패턴은 **"그 요약을 어떻게 효율적으로 만들고 관리할까"** 에 대한 고민입니다.

### 4가지 메모리 전략 요약
| 전략 | 장점 | 단점 |
|------|------|------|
| **전부 누적 (Buffer)** | 구현 간단, 디테일 전부 유지 | 토큰 비용 폭발 |
| **최근 k턴만 (Window)** | 토큰 제한 | 앞 대화 다 잊음 |
| **오래된 건 요약 (Summary)** | 비용/디테일 균형 | 요약 시간+비용, 미묘한 정보 손실 |
| **Hybrid** | 실전용 | 구현 복잡 |


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w3_memory_prompt_engineering/llm_260327_memory.ipynb)


---
## Step 0–3. Colab 환경 설정 (매 세션 1회)

Colab Secrets에 `OPENAI_API_KEY`를 등록한 뒤 아래 셀을 순서대로 실행하세요.


In [ ]:
# ════════════════════════════════════════════════════════════════
# 패키지 설치
# ════════════════════════════════════════════════════════════════
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu
!pip install -q tiktoken langchain-classic
# langchain-classic: 구버전 메모리 클래스(ConversationBufferMemory 등)가 여기로 이사함

print("패키지 설치 완료!")


In [ ]:
# ════════════════════════════════════════════════════════════════
# 환경 설정 + 기본 임포트 (매 세션마다 실행)
# ════════════════════════════════════════════════════════════════
import os
import json
import time
from datetime import datetime
from google.colab import userdata

# Colab Secrets → 환경변수
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# LangChain 핵심
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# 모델 초기화 (temperature 기본값)
llm = ChatOpenAI(model="gpt-4o-mini")
MODEL = "gpt-4o-mini"

print("환경 설정 완료!")


---
## Part 1. LLM은 **Stateless** — 기억을 못한다

### 상황
바로 위에서 "내 이름은 abc입니다"라고 했는데, 다음 요청에서 "내 이름이 뭔가요?"라고 물으면 모른다고 합니다. 왜?

### 이유
OpenAI 서버(혹은 오픈소스 모델 서버)는 **매 요청을 독립적으로** 처리해요.
- 우리가 보낸 `messages` 리스트만 읽고
- 답을 돌려주고 **즉시 잊어버림**
- 서버에 "이 사용자의 이전 대화"를 저장해 두지 않음

### 비유
LLM은 **"매번 첫 손님 응대하는 콜센터 상담원"** 이에요.
어제 내가 상담받은 내용? 상담원 입장에선 **지금 처음 듣는 얘기**.
그래서 **내가(프롬프트가) 과거 대화 요약을 가져다 붙여줘야** 대화가 이어집니다.

### 원리
매 요청마다 `messages = [전체 대화 히스토리]`를 같이 보내야 해요.
→ 이 "히스토리를 어떻게 관리할까"가 **Memory 패턴**의 본질


In [ ]:
# 첫 번째 질문: 이름 알려주기
llm.invoke("내 이름은 abc입니다.").content


In [ ]:
# 두 번째 질문: 이름 물어보기 — LLM은 바로 위 대화를 **기억 못 함**
# 왜? 각 llm.invoke()는 완전히 독립된 요청 (stateless)
llm.invoke("내 이름은 뭔가요?.").content
# → "이름을 모릅니다" 같은 답이 나옴


### 메모리 전략 미리보기

나중에 100턴 넘어가는 대화가 됐을 때, 모든 히스토리를 전부 보낼 수는 없어요 (토큰 한계/비용).
그래서 전략이 3가지:
1. **최근 4턴만 보자!** (Window)
2. **1~10턴 → 요약, 11~20턴 → 요약...** (Summary)
3. **Hybrid** (요약 + 최근 윈도우)

어떤 방식이든 결국 **`messages`에 append해서 보내는 원리는 동일**합니다.


In [ ]:
# 메모리 전략 3가지 미리 감 잡기:
# 1턴 ~ 100턴 → 최근 4턴만 보자 (Window)
# 1턴 ~ 10턴 → 요약, 11턴 ~ 20턴 → 요약 (Summary)
# 방식은 달라도 결국 매번 messages에 append해서 보내는 원리는 동일


---
## Part 2. 가장 원시적인 방법 — 직접 `messages.append()`

매번 `HumanMessage` / `AIMessage`를 리스트에 쌓아서 통째로 보내기.
LangChain 없이도 되는 "날것" 방식이에요. Memory의 원리를 이해하는 기본.


In [ ]:
# 시스템 메시지 + 첫 사용자 메시지
messages = [
    SystemMessage(content="당신은 친절한 AI 어시스턴트입니다"),
    HumanMessage(content="내 이름은 abc입니다. 반가워요"),
]

response1 = llm.invoke(messages)
print(response1.content)


In [ ]:
# AI 응답을 AIMessage로 messages에 **추가** → 다음 요청에 히스토리로 포함됨
messages.append(AIMessage(content=response1.content))
messages.append(HumanMessage(content="내 이름은 뭔가요?"))
messages  # 지금까지의 대화 흐름이 리스트로 보임


In [ ]:
# 이제 히스토리를 포함해서 다시 요청 → 이번엔 이름을 기억함!
response2 = llm.invoke(messages)
print(response2.content)
# → "abc입니다"


---
## Part 3. 토큰 비용을 **직접 측정해 보기**

`tiktoken`은 OpenAI가 만든 토큰 카운터 라이브러리. 각 모델별로 어떻게 토큰화하는지 흉내낼 수 있어요.

### 왜 측정?
대화가 길어질수록 매 요청마다 **누적된 전체 히스토리**가 토큰으로 날아가요.
- 2턴짜리: ~50 토큰
- 10턴 후: ~수백~수천 토큰
- 이게 **매 요청마다 곱해지는 비용**

### 비유
대화 히스토리는 **"카카오톡 단톡방 스크롤"** 같아요. 맨 위부터 다시 읽으면서 상황 파악해야 하는데, 단톡이 1000개쯤 쌓이면 읽는 데만 한세월.


In [ ]:
import tiktoken

# gpt-4o-mini 토크나이저 로드
enc = tiktoken.encoding_for_model("gpt-4o-mini")

def count_tokens(messages):
    """메시지 리스트의 대략적인 토큰 수를 계산"""
    total = 0
    for msg in messages:
        total += len(enc.encode(msg.content))
        total += 4  # 메시지 구분자용 여유분 (정확한 값은 모델마다 다름)
    return total

# 샘플 대화를 누적하며 토큰 수 변화 관찰
conversation = [SystemMessage(content="당신은 친절한 AI 어시스턴트입니다")]
sample_exchanges = [
    ("내 이름은 abc입니다. 반가워요", "반가워요, abc님! 어떻게 도와드릴까요?"),
    ("내 이름은 뭔가요?", "당신의 이름은 abc입니다! 다른 질문이 있으신가요?"),
]

for i, (user_msg, ai_msg) in enumerate(sample_exchanges):
    conversation.append(HumanMessage(user_msg))
    conversation.append(AIMessage(ai_msg))
    tokens = count_tokens(conversation)
    print(f"{i} | {tokens} tokens")
# → 턴을 거듭할수록 토큰이 거의 2배씩 늘어나는 게 보임


### 여기서 문제 인식
- 계속 누적 = 비용 폭증 + context length 초과
- 그래서 **Window / Summary** 같은 전략이 나왔다

#### Window
대화 윈도우 = 최신 맥락만 유지 → 앞부분은 제거

#### Summarize
10턴, 100턴 단위로 요약 → 새로 대화 시작 : 디테일 유지 + 약간의 추가 토큰 비용


---
## Part 4. `ConversationBufferMemory` (구형, deprecated)

LangChain **구버전**에서 쓰던 메모리 클래스예요. 이제는 **LangGraph / 모던 패턴**에 밀려 `langchain-classic` 패키지로 분리됐지만, 개념 이해용으로 짚어볼게요.

### 왜 deprecated?
- 내부적으로 몰래 메모리를 쥐고 있다가 꺼내 씀 → **개발자가 상태를 추적하기 어려움**
- LangGraph는 "모든 파이프(체인 구성요소)를 state로 명시" → 내부의 의도치 않은 summarize를 방지하려는 철학

### 비유
**"상담원 마음대로 메모하는 블랙박스 노트"** vs **"모든 메모가 투명하게 공유되는 회의록"**
LangChain은 이제 후자(LCEL + RunnableWithMessageHistory)로 가고 있어요.


In [ ]:
# 구버전 메모리 클래스 import (이제 langchain-classic에 있음)
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory
# DeprecationWarning이 뜨지만, 학습용이니까 일단 사용


In [ ]:
# LangGraph로 가면서 없어진 이유 정리:
# - 내부적으로 memory를 숨겨 갖고 있다가 사용 (summarize 포함)
# - LangGraph는 모든 knowledge/pipe를 state로 간주 → 개발자가 투명하게 통제
# - 그래서 ConversationBufferMemory 같은 "숨겨진 메모리"는 은퇴 중


In [ ]:
# return_messages=True → 메시지 객체 리스트로 반환 (추천)
# return_messages=False → 문자열 한 줄로 반환
memory = ConversationBufferMemory(return_messages=True)


In [ ]:
# save_context(input_dict, output_dict)
# 키 이름은 상관없음 (내부가 순서대로 Human/AI로 처리)
memory.save_context(
    {"input": "안녕하세요, 저는 abc입니다"},
    {"output": "안녕하세요, abc님! 만나서 반갑습니다"},
)
memory.save_context(
    {"input": "오늘 날씨가 좋네요!"},
    {"output": "네, 정말 화창한 날씨입니다"},
)


In [ ]:
# 저장된 히스토리 불러오기
history = memory.load_memory_variables({})


In [ ]:
history
# {'history': [HumanMessage(...), AIMessage(...), ...]}
# return_messages=True 라서 객체로 나옴


In [ ]:
# 메시지 순회 — content만 출력
for msg in history["history"]:
    print(msg.content)


In [ ]:
# return_messages=False 로 써보면 → 하나의 긴 문자열로 반환됨
memory = ConversationBufferMemory(return_messages=False)
memory.save_context(
    {"input": "안녕하세요, 저는 abc입니다"},
    {"output": "안녕하세요, abc님! 만나서 반갑습니다"},
)
memory.save_context(
    {"input": "오늘 날씨가 좋네요!"},
    {"output": "네, 정말 화창한 날씨입니다"},
)
history = memory.load_memory_variables({})
history  # 'history' 키에 "Human: ...\nAI: ...\n..." 형태 문자열
# True vs False 차이는 **포맷 차이**뿐


---
## Part 5. **모던 LangChain** — `InMemoryChatMessageHistory`

이제 권장되는 방식. `langchain_core.chat_history`에서 가져와요.

### ConversationBufferMemory와 뭐가 다른가?
사용법은 거의 동일해 보이지만, **핵심은 "Runnable과 결합"**. 모던 LangChain은 모든 컴포넌트를 `Runnable`(LCEL 체인 조각)로 다루려고 해요.
- `RunnablePassthrough`, `RunnableLambda`, `RunnableParallel`, `RunnableBranch` ...
- 그리고 오늘의 주인공 **`RunnableWithMessageHistory`**

### 비유
- 구버전 Buffer = **회의록을 상담원이 혼자 몰래 적음**
- 모던 InMemoryChatMessageHistory + Runnable = **회의록이 파이프라인의 공식 state**

겉보기엔 똑같지만, 체인 전체에 **투명하게 흘러가게** 만들어진 자료구조.


In [ ]:
# 모던 방식 import — langchain_core.chat_history
from langchain_core.chat_history import InMemoryChatMessageHistory


In [ ]:
# 초기화 후 user/ai 메시지 추가 (add_user_message / add_ai_message)
chat_history = InMemoryChatMessageHistory()
chat_history.add_user_message("안녕하세요, 저는 abc입니다")
chat_history.add_ai_message("안녕하세요, abc님! 만나서 반갑습니다")
chat_history.add_user_message("오늘 날씨가 좋네요")
chat_history.add_ai_message("네, 정말 화창한 날씨입니다")


In [ ]:
chat_history  # InMemoryChatMessageHistory 객체


In [ ]:
# 메시지 순회 — type(human/ai) + content
for msg in chat_history.messages:
    print(msg.type, msg.content)


In [ ]:
chat_history.messages  # HumanMessage/AIMessage 객체 리스트


In [ ]:
# 여기에 새로운 질문을 이어 붙여서 llm에 전달하는 첫 번째 방식
chat_history.messages.append(HumanMessage(content="내 이름이 뭔가요?"))
chat_history.messages


In [ ]:
# 그대로 넣고 호출
llm.invoke(chat_history.messages)


### 더 권장되는 패턴: 별 표(`*`)로 언패킹

`chat_history.messages`를 직접 `.append()`해서 쓰면 히스토리가 오염돼요.
별도 리스트로 분리하는 게 깔끔:

```python
messages = [
    *chat_history.messages,      # 기존 히스토리를 요소들로 펼침
    HumanMessage(content="내 이름이 뭔가요?"),
]
```
`*`는 리스트를 **엘리먼트들로 풀어서** 넣는 파이썬 문법이에요.


In [ ]:
# * 별 표 문법으로 히스토리 + 새 질문을 조합 (히스토리 오염 방지)
messages = [
    *chat_history.messages,                          # 기존 대화를 펼쳐 넣음
    HumanMessage(content="내 이름이 뭔가요?"),
]
llm.invoke(messages)


---
## Part 6. **LCEL 완성형** — `RunnableWithMessageHistory`

앞서 본 방식은 여전히 "매 요청마다 히스토리 직접 관리"여서 귀찮아요.
LCEL 체인(`prompt | llm`)에 **세션별 메모리를 자동으로 주입**해 주는 래퍼가 있습니다.

### 핵심 구조
```
prompt = ChatPromptTemplate.from_messages([
    ("system", "..."),
    MessagesPlaceholder(variable_name="history"),  # ← 여기 히스토리가 자동 주입
    ("human", "{input}"),
])
chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,       # session_id → InMemoryChatMessageHistory 반환 함수
    input_messages_key="input",
    history_messages_key="history",
)

# 사용
config = {"configurable": {"session_id": "user_001"}}
chain_with_history.invoke({"input": "..."}, config=config)
```

### 비유
- 이전 방식 = **"매번 회의록 복사해서 들고 들어가기"**
- `RunnableWithMessageHistory` = **"회의실에 들어가면 자동으로 회의록이 탁자 위에 놓여 있음"**

`session_id`는 **"누구의 회의록인지"** 알려주는 키. 유저ID/세션ID로 쓰면 유저별 기억을 분리할 수 있어요.


In [ ]:
# Runnable 생태계 간단 정리:
# - RunnablePassthrough: 입력 그대로 통과
# - RunnableLambda: 일반 함수를 체인에 끼우기
# - RunnableParallel: 여러 체인 병렬 실행
# - RunnableWithMessageHistory: 체인에 세션별 메모리 자동 연결 (오늘의 주인공)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory


In [ ]:
# 프롬프트: system + history(자동 주입) + human(사용자 입력)
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다"),
    MessagesPlaceholder(variable_name="history"),  # 이 자리에 세션 히스토리가 들어감
    ("human", "{input}"),
])

# 기본 체인 (메모리 없이)
chain = prompt | llm


In [ ]:
# 세션 저장소 — 딕셔너리: {session_id: InMemoryChatMessageHistory()}
store = {}
# 예) store = {'session_id1': ..., 'session_id2': ..., 'session_id3': InMemoryChatMessageHistory()}

def get_session_history(session_id):
    """session_id로 히스토리 찾기. 없으면 새로 만듦."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 체인에 메모리 연결
chain_with_history = RunnableWithMessageHistory(
    chain,                                  # 원본 체인
    get_session_history,                    # 세션 조회 함수
    input_messages_key="input",             # prompt의 사용자 입력 변수명
    history_messages_key="history",         # prompt의 히스토리 플레이스홀더 이름
)


In [ ]:
# 첫 번째 호출 — config로 session_id 지정 → 새 세션 히스토리 생성
config = {"configurable": {"session_id": "user_001"}}
r1 = chain_with_history.invoke({"input": "안녕하세요, 제 이름은 abc입니다"}, config=config)


In [ ]:
r1  # AIMessage 객체 — 내부적으로 store["user_001"]에 대화가 자동 저장됨


In [ ]:
# 두 번째 호출 — 같은 session_id → 히스토리가 자동으로 prompt에 주입됨
r2 = chain_with_history.invoke({"input": "내 이름이 뭐라고 했죠?"}, config=config)


In [ ]:
r2  # → "abc입니다" 같은 답. 히스토리 직접 건드린 적 없는데 기억함!


In [ ]:
# store 구경 — user_001 세션에 대화가 쌓여 있음
store


In [ ]:
# 한 번 더 이어가기
r3 = chain_with_history.invoke({"input": "오늘 뭐하면 좋을까요?"}, config=config)


In [ ]:
r3.content


### 실습: 전자제품 판매원 챗봇

같은 패턴으로 다른 시스템 프롬프트의 챗봇을 만들어 봐요.
(`store`를 새로 만들어서 user_001 세션을 초기화)


In [ ]:
# 전자제품 판매원 버전
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 전자제품 판매원입니다"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
chain = prompt | llm

# 새 store로 초기화 (user_001 재사용)
store = {}
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "user_001"}}
r1 = chain_with_history.invoke({"input": "안녕하세요, 노트북을 사려는데 어떤게 있나요?"}, config=config)


In [ ]:
r1.content


In [ ]:
# 맥락 이어서 — 같은 세션이라 "인기 제품" = 위에서 추천한 제품으로 이해
r2 = chain_with_history.invoke({"input": "가장 인기 있는 제품의 가격이 어떻게 되나요?"}, config=config)


In [ ]:
r2.content


In [ ]:
# "2번째 제품" 같은 지시대명사도 히스토리 덕에 이해함
r3 = chain_with_history.invoke({"input": "좋아요, 2번째 제품으로 구입하겠습니다."}, config=config)
r3.content


In [ ]:
# 세션에 쌓인 메시지 개수 — 요청 3회 = user/ai 쌍 3쌍 = 6개
len(store["user_001"].messages)


In [ ]:
# 세션 메시지 순회 — type으로 customer/sales person 태그 표시
for msg in store["user_001"].messages:
    prefix = "customer" if msg.type == "human" else "sales person"
    print(f"[{prefix}] - {msg.content[:30]}")


---
## Part 7. **Window 메모리** — 최근 k턴만 유지

전부 쌓으면 토큰 폭발. 그래서 **최근 k턴(또는 k쌍)만** 보관하는 전략.

### 구버전: `ConversationBufferWindowMemory(k=2)`
역시 **deprecated**. 하지만 개념 확인용.

### 모던 버전: `trim_messages()`
`langchain_core.messages`에 있는 유틸 함수.
```python
trimmer = trim_messages(max_tokens=4, strategy="last", token_counter=len, start_on="human")
trimmed = trimmer.invoke(messages)
```
- `strategy="last"`: 최신에서부터 남김
- `token_counter=len`: 토큰 대신 **메시지 개수**로 카운트 (개수 기반 자르기)

### 그냥 파이썬 슬라이싱으로도 OK
`messages[-(K*2):]` 같은 식으로 마지막 K턴(=K*2 메시지) 남기기. 실전에선 오히려 이게 명료.

### 비유
Window 메모리는 **"카톡 방을 열면 최근 20개만 스크롤에 보이는"** 느낌.
이전 대화는 영원히 사라짐. 빠르고 싸지만 **맥락 유지 약함**.


In [ ]:
# 구버전 — k=2: 최근 2쌍(4메시지)만 유지
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory

window_memory = ConversationBufferWindowMemory(k=2, return_messages=True)
window_memory.save_context({"input": "첫번째, 내 이름은 abc입니다"}, {"output": "안녕하세요, abc님"})
window_memory.save_context({"input": "두번째, 나는 학생입니다"}, {"output": "학생이시군요! 멋지십니다"})
window_memory.save_context({"input": "세번째, 나는 파이썬을 좋아합니다"}, {"output": "파이썬은 정말 재미있죠!"})


In [ ]:
# k=2 이므로 **첫번째 대화는 날아가고** 최근 2쌍만 남음
window_memory.load_memory_variables({})


In [ ]:
# 모던 버전 — trim_messages
from langchain_core.messages import trim_messages

messages = []
conversations = [
    ("첫번째, 내 이름은 abc입니다", "안녕하세요, abc님"),
    ("두번째, 나는 학생입니다", "학생이시군요! 멋지십니다"),
    ("세번째, 나는 파이썬을 좋아합니다", "파이썬은 정말 재미있죠!"),
]

for user_msg, ai_msg in conversations:
    messages.append(HumanMessage(content=user_msg))
    messages.append(AIMessage(content=ai_msg))

messages


In [ ]:
# max_tokens=4: 개수 4개까지 (쌍 2개)
# strategy="last": 최신부터 남김
# token_counter=len: 메시지 개수 기반
# start_on="human": human 메시지부터 시작하도록 보장
trimmer = trim_messages(max_tokens=4, strategy="last", token_counter=len, start_on="human")
trimmed = trimmer.invoke(messages)


In [ ]:
trimmed  # 최근 4개 메시지 = 최근 2쌍만 남음


In [ ]:
# 직접 만들기 — 그냥 리스트 슬라이싱이 가장 명료
messages = []
K = 2

def add_turn(user_input, ai_output):
    global messages  # 함수 밖의 messages에 접근
    messages.append(HumanMessage(content=user_input))
    messages.append(AIMessage(content=ai_output))
    # 마지막 K쌍(=K*2개)만 남기고 나머지 버림
    messages = messages[-(K * 2):]

# 세 턴을 차례로 추가하며 messages가 어떻게 잘리는지 관찰
add_turn("첫번째, 내 이름은 abc입니다", "안녕하세요, abc님")
print("11111")
print(messages)

add_turn("두번째, 나는 학생입니다", "학생이시군요! 멋지십니다")
print("22222")
print(messages)

add_turn("세번째, 나는 파이썬을 좋아합니다", "파이썬은 정말 재미있죠!")
print("33333")
print(messages)
# → 세번째 턴에서 첫번째 대화가 슬라이싱으로 사라짐


---
## Part 8. **Summary 메모리** — 오래된 건 LLM이 요약

### 왜 Summary?
- Window = 앞 대화 **그냥 버림** → 맥락 손실 심함
- Summary = 앞 대화를 **LLM에 요약시켜 압축** → 디테일 일부 손실 + 약간의 추가 토큰 비용

### 트레이드오프
| | 장점 | 단점 |
|---|------|------|
| Window | 빠름, 비용 예측 가능 | 오래된 맥락 소실 |
| Summary | 오래된 맥락 보존 | **요약 호출 = 추가 LLM 비용 + 지연** |

### 구버전: `ConversationSummaryMemory`
**deprecated**. 이유:
- 요약용 LLM 따로 init해야 함
- "언제 어떻게 요약할지" **블랙박스** → 개발자가 state 통제 어려움
- 요즘은 **직접 요약 로직 작성** 추천

### 비유
Summary 메모리 = **"상담원이 퇴근 전 오늘 상담 내용을 한 문장으로 요약해서 다음 날 인수인계"**.
편한데, 요약하다가 디테일 빠지면 다음 날 "아 그게 뭐였더라…" 하는 일이 생김.


In [ ]:
# 구버전 ConversationSummaryMemory — deprecated지만 개념 확인
from langchain_classic.memory import ConversationSummaryMemory


In [ ]:
# 요약용 LLM을 별도로 생성 (temperature 낮게)
summary_llm = ChatOpenAI(model="gpt-4o-mini")
summary_memory = ConversationSummaryMemory(
    llm=summary_llm,
    return_messages=False,  # 문자열 반환
)

conversations = [
    ("첫번째, 내 이름은 abc입니다", "안녕하세요, abc님"),
    ("두번째, 나는 학생입니다", "학생이시군요! 멋지십니다"),
    ("세번째, 나는 파이썬을 좋아합니다", "파이썬은 정말 재미있죠!"),
]

# 1. llm 호출이 들어가므로 시간이 오래 걸림
# 2. DeprecationWarning 뜸 — 구버전이라 그래요
for user_msg, ai_msg in conversations:
    summary_memory.save_context({"input": user_msg}, {"output": ai_msg})


In [ ]:
# 요약된 히스토리 꺼내기
result = summary_memory.load_memory_variables({})


In [ ]:
result["history"]  # 전체 대화를 한 문단으로 요약한 문자열
# → "사용자가 이름은 abc이고 학생이며 파이썬을 좋아한다고..." 같은 축약


---
## Part 9. **직접 만드는 Summary 메모리** — 투명하게

구버전 `ConversationSummaryMemory`가 마음에 안 드는 이유가 **블랙박스**라면, **직접 만드는 게 답**. 원하는 시점에 요약, 원하는 프롬프트로 요약.

### 설계
- `summary`: 지금까지의 대화 요약 (누적)
- `recent_messages`: 요약되지 않은 최신 대화들
- `summary_interval`: **N턴마다** 요약 실행 (매턴 요약은 너무 비쌈)
- `turn_count`: 턴 카운트
- `add_exchange(user_msg, ai_msg)`: 대화 추가. interval이 되면 자동 요약
- `_summarize(text)`: LLM 호출해서 요약 (기존 요약 + 새 대화 병합)
- `get_context()`: 현재 컨텍스트 반환 = `[요약] + [최근 메시지]`

### 비유
**"가계부"** 같은 느낌이에요.
- 매 거래마다 기록 (recent_messages)
- 매달 말일에 "이번 달 총 지출 X원, 카테고리별 비중..." 한 줄 요약 (summary)
- 필요할 땐 **요약 + 이번 달 거래내역** 둘 다 봄 (get_context)


In [ ]:
class SummaryMemory:
    """N턴마다 대화를 요약하는 커스텀 메모리"""

    def __init__(self, summary_interval=3):
        self.summary = ""                    # 누적 요약 (문자열)
        self.recent_messages = []            # 아직 요약 안 된 최근 메시지들
        self.summary_interval = summary_interval  # N턴마다 요약
        self.turn_count = 0                  # 누적 턴 카운트

    def _summarize(self, text):
        """기존 요약 + 새 대화 → LLM에 요약 요청"""
        response = llm.invoke([
            SystemMessage(content="주어진 대화 내용을 핵심만 간결하게 요약하세요. 한국어로 작성하세요"),
            HumanMessage(content=f"기존 요약:\n{self.summary}\n\n새 대화:\n{text}"),
        ])
        return response.content

    def add_exchange(self, user_msg, ai_msg):
        """user/ai 쌍 추가. interval이 되면 자동 요약"""
        self.recent_messages.append(f"사용자 : {user_msg}")
        self.recent_messages.append(f"AI : {ai_msg}")
        self.turn_count += 1

        # summary_interval 배수마다 요약 실행
        if self.turn_count % self.summary_interval == 0:
            conversation_text = "\n".join(self.recent_messages)
            self.summary = self._summarize(conversation_text)
            self.recent_messages = []  # 요약 후 비움
            print(f" 요약 완료 턴 {self.turn_count} 에서 요약")

    def get_context(self):
        """요약 + 최근 대화를 합쳐서 반환 (프롬프트에 넣을 컨텍스트)"""
        parts = []
        if self.summary:
            parts.append(f"[이전 대화 요약] {self.summary}")
        if self.recent_messages:
            parts.append(f"[최근 대화]\n" + "\n".join(self.recent_messages))
        return "\n\n".join(parts)


In [ ]:
# interval=2: 2턴마다 요약
smem = SummaryMemory(summary_interval=2)


In [ ]:
# 3턴을 추가하며 요약이 작동하는지 관찰
conversations = [
    ("첫번째, 내 이름은 abc입니다", "안녕하세요, abc님"),
    ("두번째, 나는 학생입니다", "학생이시군요! 멋지십니다"),
    ("세번째, 나는 파이썬을 좋아합니다", "파이썬은 정말 재미있죠!"),
]

for user_msg, ai_msg in conversations:
    smem.add_exchange(user_msg, ai_msg)
# → 2턴째에서 " 요약 완료 턴 2 에서 요약" 출력됨


In [ ]:
# 현재 컨텍스트 확인 — [요약] + [최근 대화] 형태
smem.get_context()
# 문제: "내 이름 뭐야?" 라고 물어보면 요약 속에 이름이 있어야 답 가능
# → 요약 프롬프트를 "이름/직업/관심사 같은 키 정보는 반드시 유지" 식으로 다듬어야 함


---
## Part 10. **실전 챗봇** — `SummaryChatbot` 클래스

`SummaryMemory`를 실제 챗봇으로 감싸보자. 특정 시스템 프롬프트(예: IT 커리어 상담사)로 대화하고, 내부적으로는 요약 메모리로 토큰 비용 통제.

### 구조
```
SummaryChatbot
 ├─ system_prompt
 ├─ memory: SummaryMemory
 └─ chat(user_input):
     1. memory.get_context() → 이전 대화 요약+최근
     2. [system, (context), human] 메시지 조립
     3. llm.invoke()
     4. memory.add_exchange(user_input, ai_response)
     5. return ai_response
```

### 비유
**"IT 커리어 상담사 봇"** 을 고용했는데, 상담 내용은 매 턴 기록되고, **2턴마다 요약이 자동**으로 이루어지는 구조.
상담사(봇)는 요약 + 최근 대화만 보고 다음 답을 해줌.


In [ ]:
class SummaryChatbot:
    def __init__(self, system_prompt="당신은 도움이 되는 AI 어시스턴트입니다.", summary_interval=3):
        self.system_prompt = system_prompt
        self.memory = SummaryMemory(summary_interval=summary_interval)

    def chat(self, user_input):
        # 1) 현재까지의 컨텍스트(요약+최근) 가져오기
        context = self.memory.get_context()

        # 2) 메시지 조립: system → (있으면) context → human
        messages = [SystemMessage(content=self.system_prompt)]
        if context:
            messages.append(SystemMessage(content=f"대화 맥락 : \n{context}"))
        messages.append(HumanMessage(content=user_input))

        # 3) LLM 호출
        response = llm.invoke(messages)
        ai_response = response.content

        # 4) 이번 턴을 메모리에 기록 (interval이면 자동 요약됨)
        self.memory.add_exchange(user_input, ai_response)

        return ai_response


In [ ]:
# IT 커리어 상담사 페르소나로 세팅, 2턴마다 요약
bot = SummaryChatbot(
    system_prompt="당신은 IT 커리어 상담사입니다",
    summary_interval=2,
)


In [ ]:
# 4개 질문 시나리오 — 중간에 요약이 발생함
questions = [
    "안녕하세요, 백엔드 개발자 3년차인데 고민이 있습니다",
    "AI/ML 분야로 전환을 고려 중인데 어떤 준비가 필요할까요?",
    "현재 Python은 잘하는데 수학과 통계 기초가 부족합니다",
    "온라인 강의와 학교 중 어떤 것이 효과적일까요?",
]


In [ ]:
# 전체 실행 — [user] 질문, [상담사] 답변이 교차 출력
# 2턴째, 4턴째에서 "요약 완료" 메시지가 사이사이 찍힘
for q in questions:
    print(f"[user] {q}")
    answer = bot.chat(q)
    print(f"[상담사] {answer}")
    print()


---
## 오늘의 정리

### LLM은 Stateless — 매 요청 독립
히스토리 관리는 **개발자 책임**. 매 요청에 과거 맥락을 담아야 대화 유지.

### 메모리 전략 4가지 비교
| 전략 | 토큰 비용 | 맥락 보존 | 구현 복잡도 |
|------|-----------|-----------|-------------|
| Buffer (전부) | 높음 ↑↑ | 완벽 | 쉬움 |
| Window (최근 k) | 예측 가능 | 오래된 것 손실 | 쉬움 |
| Summary | 중간 (요약 비용) | 압축 (디테일 일부 손실) | 중간 |
| Hybrid | 최적 | 최적 | 복잡 |

### 기술 스택
- 구형: `ConversationBufferMemory`, `ConversationBufferWindowMemory`, `ConversationSummaryMemory` (deprecated)
- 모던: `InMemoryChatMessageHistory` + `RunnableWithMessageHistory`
- 유틸: `trim_messages`, `tiktoken`

### LCEL 체인에 메모리 달기 (현재 표준)
```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "..."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)
```

### 기억할 한 문장
> **"LLM은 매번 처음 보는 사이. 내가(프롬프트가) 지난 이야기를 요약해서 건네야 대화가 이어진다."**

### 다음 주 예고 (Week 4)
- RAG 성능 평가 (MRR, Hit Rate)
- 하이브리드 검색 (BM25 + 벡터)
- 재랭킹(re-ranking)
